# EchoFactory - FAN: Preprocessing v3 (Mel + TRUE T-gram)

## Perbaikan dari fanechofac2:
- ❌ **Branch 1 lama**: MFCC+Delta → kehilangan info transien anomali, embeddings tidak separable
- ✅ **Branch 1 baru**: T-gram (STFT linear tanpa mel filterbank, n_fft=512, hop=256)
  - Resolusi frekuensi lebih tinggi (512 bins vs 40 MFCC)
  - Menangkap fine-grained frequency details yang anomali FAN munculkan
  - Korelasi rendah dengan Mel → dual-branch saling melengkapi


In [ ]:
import os, gc, glob, json
import numpy as np
import torch
import torch.nn.functional as F
import librosa
from tqdm.auto import tqdm

# =============================================
# KONFIGURASI — Sesuaikan jika path berbeda
# =============================================
DATASET_ROOT  = '/kaggle/input/datasets/bisheshgiri/mimii-dataset'
OUT_DIR       = '/kaggle/working'
TARGET_SNR    = '0_dB'
MACHINE_TYPES = ['fan']
SR            = 16000
IMG_SIZE      = (128, 128)

# Branch 0: Mel Spectrogram (sama seperti sebelumnya)
MEL_N_MELS, MEL_N_FFT, MEL_HOP = 128, 1024, 512

# Branch 1: T-gram (STFT linear — BARU)
# Menggunakan STFT tanpa mel filterbank untuk resolusi frekuensi yang lebih tinggi
# n_fft=512 → 257 bins, kita ambil 128 bins pertama (0–4kHz)
TG_N_FFT, TG_HOP = 512, 256

os.makedirs(f'{OUT_DIR}/features_v3', exist_ok=True)
print('Config OK | Dual-branch v3: [Mel Spectrogram] + [T-gram STFT linear]')
print(f'Dataset: {DATASET_ROOT}')


In [ ]:
def compute_mel(wav):
    """Branch 0: Log-Mel Spectrogram. Sama seperti sebelumnya."""
    mel = librosa.feature.melspectrogram(
        y=wav, sr=SR, n_mels=MEL_N_MELS, n_fft=MEL_N_FFT, hop_length=MEL_HOP
    )
    return librosa.power_to_db(mel, ref=np.max)  # shape: (128, T)

def compute_tgram_v3(wav):
    """
    Branch 1: T-gram — STFT linear tanpa mel filterbank.
    
    KENAPA STFT linear lebih baik dari MFCC+Delta untuk FAN anomaly detection:
    - Anomali pada FAN (bearing fault, imbalance) muncul sebagai narrow-band frequency spikes
    - Mel filterbank meratakan/mengaburkan spike ini dengan triangular averaging
    - STFT linear mempertahankan frekuensi individu dengan presisi penuh
    - n_fft=512 (hop=256) → resolusi waktu 16ms, cukup untuk transient anomaly
    """
    # STFT magnitude
    S = np.abs(librosa.stft(wav, n_fft=TG_N_FFT, hop_length=TG_HOP))  # (257, T)
    # Ambil 128 bins pertama (0–4kHz untuk 16kHz audio)
    S = S[:128, :]  # (128, T)
    # Convert ke dB
    S = librosa.amplitude_to_db(S, ref=np.max)  # (128, T)
    return S

def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

def to_fixed_tensor(x_np, size=IMG_SIZE):
    """Resize ke (128, 128) menggunakan bilinear interpolation."""
    t = torch.FloatTensor(x_np).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    return F.interpolate(t, size=size, mode='bilinear', align_corners=False).squeeze(0)  # (1, 128, 128)

def extract_features_v3(wav_path):
    wav, _ = librosa.load(wav_path, sr=SR, mono=True)
    mel_feat = to_fixed_tensor(normalize(compute_mel(wav)))       # (1, 128, 128)
    tg_feat  = to_fixed_tensor(normalize(compute_tgram_v3(wav)))  # (1, 128, 128)
    return torch.cat([mel_feat, tg_feat], dim=0)  # (2, 128, 128)

# === VERIFIKASI ===
test_files = glob.glob(f'{DATASET_ROOT}/0_dB_fan/fan/id_00/normal/*.wav')
if test_files:
    feat = extract_features_v3(test_files[0])
    print(f'Feature shape: {feat.shape}  (expected: [2, 128, 128])')
    print(f'Branch 0 (Mel):    mean={feat[0].mean():.3f}, std={feat[0].std():.3f}')
    print(f'Branch 1 (T-gram): mean={feat[1].mean():.3f}, std={feat[1].std():.3f}')
    corr = torch.corrcoef(torch.stack([feat[0].flatten(), feat[1].flatten()]))[0, 1]
    print(f'Correlation antar branch: {corr:.3f}  (target: < 0.5)')
    if abs(corr) < 0.7:
        print('✓ Korelasi rendah — dual-branch efektif!')
    else:
        print('⚠ Korelasi tinggi — pertimbangkan ulang fitur')
else:
    print(f'⚠ Tidak ada file WAV ditemukan di: {DATASET_ROOT}/0_dB_fan/fan/id_00/normal/')
    print('  Pastikan dataset sudah di-add sebagai input Kaggle')


In [ ]:
import matplotlib.pyplot as plt

if test_files:
    # Visualisasi perbandingan branch
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].imshow(feat[0].numpy(), aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title('Branch 0: Mel Spectrogram', fontweight='bold')
    axes[0].set_ylabel('Mel Bins'); axes[0].set_xlabel('Frames')
    
    axes[1].imshow(feat[1].numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title('Branch 1: T-gram (STFT Linear)', fontweight='bold')
    axes[1].set_ylabel('Frequency Bins (0–4kHz)'); axes[1].set_xlabel('Frames')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/feature_branches_v3.png', dpi=150)
    plt.show()
    print('Kedua branch HARUS terlihat berbeda (textural vs. structural)')


In [ ]:
# === BUILD LABEL MAPS ===
label_maps = {}
for m in MACHINE_TYPES:
    mp = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{m}', m)
    if not os.path.exists(mp):
        print(f'SKIP: {mp} tidak ada'); continue
    ids = sorted(os.listdir(mp))
    label_maps[m] = {mid: i for i, mid in enumerate(ids)}
    print(f'  {m}: {label_maps[m]}')

with open(f'{OUT_DIR}/label_maps_v3.json', 'w') as f:
    json.dump(label_maps, f, indent=2)
print('Label maps saved.')


In [ ]:
# === EKSTRAK SEMUA FITUR ===
for machine in MACHINE_TYPES:
    print(f'\nProcessing {machine.upper()}...')
    lmap = label_maps.get(machine, {})
    mp   = os.path.join(DATASET_ROOT, f'{TARGET_SNR}_{machine}', machine)
    if not os.path.exists(mp):
        print(f'  SKIP: {mp} tidak ada'); continue

    for condition in ['normal', 'abnormal']:
        all_feats, all_labels, all_paths = [], [], []
        for mid, label_id in lmap.items():
            cp    = os.path.join(mp, mid, condition)
            if not os.path.exists(cp): continue
            files = sorted(glob.glob(os.path.join(cp, '*.wav')))
            print(f'  {mid}/{condition}: {len(files)} files')
            for fp in tqdm(files, desc=f'{mid}/{condition}', leave=False):
                try:
                    all_feats.append(extract_features_v3(fp))
                    all_labels.append(label_id)
                    all_paths.append(fp)
                except Exception as e:
                    print(f'  Skip {os.path.basename(fp)}: {e}')

        if not all_feats:
            print(f'  Tidak ada fitur untuk {machine}/{condition}'); continue

        feats_tensor = torch.stack(all_feats).to(torch.float32)
        save_path = f'{OUT_DIR}/features_v3/{machine}_{condition}.pt'
        torch.save({
            'features': feats_tensor,
            'labels': torch.LongTensor(all_labels),
            'paths': all_paths
        }, save_path)
        mb = os.path.getsize(save_path) / (1024**2)
        print(f'  SAVED: {save_path} | Shape: {feats_tensor.shape} | {mb:.1f} MB')
        del all_feats, all_labels, all_paths, feats_tensor
        gc.collect()

    print(f'{machine.upper()} DONE')

print('\nfanechofac6 SELESAI! → Lanjut ke fanechofac7 (Training v3)')
